In [1]:
import os
import json
import duckdb
import hashlib

# Setup paths
FHIR_DIR = "../synthea/output/fhir"
DB_PATH = "../data/omop_clinical.duckdb"

def stable_person_id(source_id: str) -> int:
    """
    Generates a stable, deterministic integer ID from a string using SHA-256.
    Crucial for maintaining relational integrity with the PERSON table.
    """
    return int(hashlib.sha256(source_id.encode()).hexdigest(), 16) % (10**9)

def extract_conditions(patient_file):
    """
    Reads a FHIR bundle and extracts condition (diagnosis) records.
    Returns a list of tuples ready for bulk database insertion.
    """
    file_path = os.path.join(FHIR_DIR, patient_file)
    with open(file_path, 'r', encoding='utf-8') as f:
        fhir_data = json.load(f)
        
    conditions = []
    
    for entry in fhir_data.get('entry', []):
        resource = entry.get('resource', {})
        
        if resource.get('resourceType') == 'Condition':
            
            # CRITICAL FIX 1: Deterministic Person ID
            subject_ref = resource.get('subject', {}).get('reference', '')
            patient_source_id = subject_ref.replace('urn:uuid:', '')
            person_id = stable_person_id(patient_source_id)
            
            # Extract SNOMED Code and Text Description
            coding = resource.get('code', {}).get('coding', [])
            snomed_code = "0"
            condition_text = "Unknown"
            
            if coding:
                snomed_code = coding[0].get('code', '0')
                condition_text = coding[0].get('display', 'Unknown')
                
            # Extract Start Date
            start_date = resource.get('onsetDateTime', '1900-01-01')[:10] 
            
            conditions.append((
                person_id,
                snomed_code,
                condition_text,
                start_date
            ))
            
    return conditions

print("⚙️ STARTING ETL PIPELINE (FHIR -> OMOP CONDITION) [V2: STABLE & IDEMPOTENT]\n" + "-"*50)

print("🔍 Extracting diagnoses from FHIR JSON files...")
json_files = [f for f in os.listdir(FHIR_DIR) if f.endswith('.json')]
all_conditions = []

for file in json_files:
    all_conditions.extend(extract_conditions(file))

print(f"📊 Extracted {len(all_conditions)} raw clinical conditions.")
print("🔌 Connecting to DuckDB for vocabulary mapping...")

try:
    with duckdb.connect(DB_PATH) as con:
        
        con.execute("DROP TABLE IF EXISTS stg_condition")
        con.execute("""
            CREATE TEMPORARY TABLE stg_condition (
                person_id BIGINT,
                snomed_code VARCHAR,
                condition_text VARCHAR,
                start_date DATE
            )
        """)
        
        con.executemany("""
            INSERT INTO stg_condition VALUES (?, ?, ?, ?)
        """, all_conditions)
        
        print("⏳ Creating standard OMOP CONDITION_OCCURRENCE table...")
        
        # CRITICAL FIX 3: condition_source_concept_id is now INTEGER per OMOP specs
        con.execute("""
            CREATE TABLE IF NOT EXISTS condition_occurrence (
                condition_occurrence_id BIGINT PRIMARY KEY,
                person_id BIGINT,
                condition_concept_id INTEGER,
                condition_start_date DATE,
                condition_source_value VARCHAR,
                condition_source_concept_id INTEGER 
            )
        """)
        
        # CRITICAL FIX 2: Idempotency (prevent duplicates on re-runs)
        print("🧹 Cleaning existing data to prevent duplicates (Idempotency)...")
        con.execute("DELETE FROM condition_occurrence")
        
        print("🧠 Performing SQL JOIN with OMOP Concept table...")
        
        # The SQL now correctly retrieves the integer concept_id for the source code
        con.execute("""
            INSERT INTO condition_occurrence 
            SELECT 
                ROW_NUMBER() OVER () AS condition_occurrence_id,
                stg.person_id,
                COALESCE(c.concept_id, 0) AS condition_concept_id,
                stg.start_date AS condition_start_date,
                stg.condition_text AS condition_source_value,
                COALESCE(c.concept_id, 0) AS condition_source_concept_id
            FROM stg_condition stg
            LEFT JOIN concept c 
                ON stg.snomed_code = c.concept_code 
                AND c.vocabulary_id = 'SNOMED'
                AND c.domain_id = 'Condition'
        """)
        
        mapped_count = con.execute("SELECT COUNT(*) FROM condition_occurrence WHERE condition_concept_id != 0").fetchone()[0]
        unmapped_count = con.execute("SELECT COUNT(*) FROM condition_occurrence WHERE condition_concept_id = 0").fetchone()[0]
        
        print(f"\n✅ ETL Complete!")
        print(f" - Successfully mapped to OMOP Standards: {mapped_count} conditions")
        print(f" - Failed to map (Requires AI / Domain fix): {unmapped_count} conditions")
        
        # THE ULTIMATE TEST: Checking for orphaned records
        print("\n🔎 Verifying relational integrity (JOIN with PERSON table)...")
        orphans = con.execute("""
            SELECT COUNT(*) FROM condition_occurrence c 
            LEFT JOIN person p ON c.person_id = p.person_id 
            WHERE p.person_id IS NULL
        """).fetchone()[0]
        
        if orphans == 0:
            print("🔗 PERFECT INTEGRITY: All conditions belong to a valid Person ID!")
        else:
            print(f"⚠️ WARNING: Found {orphans} orphaned conditions! The bridge is broken.")

except Exception as e:
    print(f"❌ Database error: {e}")

⚙️ STARTING ETL PIPELINE (FHIR -> OMOP CONDITION) [V2: STABLE & IDEMPOTENT]
--------------------------------------------------
🔍 Extracting diagnoses from FHIR JSON files...
📊 Extracted 1758 raw clinical conditions.
🔌 Connecting to DuckDB for vocabulary mapping...
⏳ Creating standard OMOP CONDITION_OCCURRENCE table...
🧹 Cleaning existing data to prevent duplicates (Idempotency)...
🧠 Performing SQL JOIN with OMOP Concept table...

✅ ETL Complete!
 - Successfully mapped to OMOP Standards: 680 conditions
 - Failed to map (Requires AI / Domain fix): 1078 conditions

🔎 Verifying relational integrity (JOIN with PERSON table)...
🔗 PERFECT INTEGRITY: All conditions belong to a valid Person ID!
